<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Tokenization/Candlestick_BPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Documentation
This notebook demonstrates a methodology for identifying high-conviction trading signals using time-series analysis and Byte-Pair Encoding (BPE) on candlestick patterns. It covers data acquisition, tokenization of candlestick data, multi-sequence BPE compression, calculation of transition matrices, optimization of trading edges, and out-of-sample validation of these signals.

In [27]:
# Install necessary libraries
#!pip install hmmlearn
!pip install tokenizers

In [28]:
# Import required libraries
import numpy as np
import pandas as pd
import yfinance as yf
#from hmmlearn import hmm
import matplotlib.pyplot as plt

from collections import Counter
import pandas as pd

In [29]:
# Define Google Drive file IDs
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

In [30]:
# Step 1: Data fetch and candlestick tokenizer

def generate_candlestick_tokens(df: pd.DataFrame) -> pd.Series:
    """Normalizes daily OHLC data and tokenizes the candlestick geometry."""
    total_range = df["High"] - df["Low"]
    body_length = df["Close"] - df["Open"]

    # Prevent division by zero
    total_range = np.where(total_range == 0, 1e-5, total_range)

    # Relative shadow calculations
    upper_wick = df["High"] - np.maximum(df["Open"], df["Close"])
    upper_wick_ratio = upper_wick / total_range

    lower_wick = np.minimum(df["Open"], df["Close"]) - df["Low"]
    lower_wick_ratio = lower_wick / total_range

    # Normalized body change relative to open price
    body_ratio = body_length / df["Open"]

    # Discrete structural mappings
    def map_body(val):
        if val <= -0.015:
            return "LL"  # Large Loss
        elif val <= -0.003:
            return "ML"  # Moderate Loss
        elif abs(val) < 0.003:
            return "DO"  # Doji
        elif val < 0.015:
            return "MG"  # Moderate Gain
        else:
            return "LG"  # Large Gain

    def map_wick(val):
        if val <= 0.15:
            return "S"  # Small
        elif val <= 0.40:
            return "M"  # Medium
        else:
            return "L"  # Large

    v_map_body = np.vectorize(map_body)
    v_map_wick = np.vectorize(map_wick)

    candle_tokens = (
        pd.Series(v_map_body(body_ratio), index=df.index)
        + pd.Series(v_map_wick(upper_wick_ratio), index=df.index)
        + pd.Series(v_map_wick(lower_wick_ratio), index=df.index)
    )

    return candle_tokens


# =====================================================================
# EXECUTION PIPELINE
# =====================================================================
if __name__ == "__main__":
    # 1. Load the list of symbols from your target file
    symbols_df = pd.read_csv(OptionVolume)
    symbols_list = symbols_df["Symbol"].dropna().unique().tolist()

    START_DATE = "2018-01-01"
    END_DATE = "2024-12-31"

    print(
        f"Downloading daily data for {len(symbols_list)} symbols from {START_DATE} to {END_DATE}..."
    )

    # 2. Multi-threaded download across the full asset matrix
    raw_data = yf.download(
        tickers=symbols_list,
        start=START_DATE,
        end=END_DATE,
        interval="1d",
        auto_adjust=True,
    )

    # 3. Stack columns from wide format to long format (Date, Ticker) row MultiIndex
    if isinstance(raw_data.columns, pd.MultiIndex):
        if "Ticker" in raw_data.columns.names:
            stacked_data = raw_data.stack(level="Ticker")
        else:
            stacked_data = raw_data.stack(level=1)
    else:
        # Fallback to keep execution working if data is only a single asset
        stacked_data = raw_data.copy()
        stacked_data["Ticker"] = "QQQ"
        stacked_data = stacked_data.set_index(["Ticker"], append=True)

    # Drop missing holiday values
    stacked_data = stacked_data.dropna(subset=["Open", "High", "Low", "Close"])

    print("Generating Candlestick Tokens across stacked multi-symbol framework...")
    stacked_data["Candle_Token"] = generate_candlestick_tokens(stacked_data)

    # 4. Reconstruct clean individual timeline lists for BPE execution
    ticker_to_sequence = {}
    for ticker, group in stacked_data["Candle_Token"].groupby(level="Ticker"):
        chronological_seq = group.droplevel("Ticker").sort_index()
        ticker_to_sequence[ticker] = chronological_seq.astype(str).tolist()

    print(
        f"\n--- Framework Initialization Complete: {len(ticker_to_sequence)} Active Timelines Set ---"
    )
    print(stacked_data[["Open", "High", "Low", "Close", "Candle_Token"]].head())

[*********************100%***********************]  100 of 100 completed
ERROR:yfinance:
3 Failed downloads:
ERROR:yfinance:['CRCL', 'SNDK', 'CRWV']: YFPricesMissingError('possibly delisted; no price data found  (1d 2018-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1514782800, endDate = 1735621200")')
/tmp/ipykernel_3057/3374415935.py:81: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked_data = raw_data.stack(level="Ticker")


Generating Candlestick Tokens across stacked multi-symbol framework...

--- Framework Initialization Complete: 97 Active Timelines Set ---
Price                    Open        High         Low       Close Candle_Token
Date       Ticker                                                             
2018-01-02 AAOI     37.799999   38.770000   37.619999   37.910000         DOLM
           AAPL     39.776190   40.276431   39.565806   40.267078         MGSM
           ADBE    175.850006  177.800003  175.259995  177.699997         MGSM
           AMAT     47.075368   48.332412   46.656354   48.314194         LGSM
           AMD      10.420000   11.020000   10.340000   10.980000         LGSS


In [31]:
# Step 2: Native multi-sequence Byte-Pair Encoding

def native_time_series_bpe(
    all_ticker_sequences: dict[str, list[str]], target_vocab_size: int = 100
) -> tuple[dict[str, list[int]], dict[int, str]]:
    """Performs deterministic Byte-Pair Encoding across multiple sequences,

    ensuring multi-day patterns are explicitly merged without hashing issues and
    globally selecting the most frequent pairs across all assets.
    """
    # 1. Initialize global vocabulary with all unique base tokens from all sequences
    unique_base_tokens = set()
    for ticker_seq in all_ticker_sequences.values():
        unique_base_tokens.update(ticker_seq)

    initial_vocab_list = sorted(list(unique_base_tokens))
    vocab = {i: tok for i, tok in enumerate(initial_vocab_list)}
    inverse_vocab = {tok: i for i, tok in vocab.items()}

    current_vocab_size = len(vocab)
    print(f"Initial unique daily candle types found: {current_vocab_size}")

    # Convert all ticker sequences to a mutable list of current tokens
    current_sequences = {ticker: list(seq) for ticker, seq in all_ticker_sequences.items()}

    # Loop until we hit our target vocabulary size or no more merges are possible
    while current_vocab_size < target_vocab_size:
        # 2. Count all consecutive pairs of tokens across ALL ticker sequences
        pair_counts = Counter()
        for ticker, sequence in current_sequences.items():
            for i in range(len(sequence) - 1):
                pair_counts[(sequence[i], sequence[i + 1])] += 1

        if not pair_counts:
            break

        # 3. Find the single most frequently occurring multi-day pair across the entire market
        best_pair, most_frequent_count = pair_counts.most_common(1)[0]

        # Guardrail: If a pattern only happens once, stop merging to avoid overfitting
        if most_frequent_count < 2:
            break

        # 4. Register the new merged token into our global vocabulary
        new_token_string = f"{best_pair[0]} ➔ {best_pair[1]}"

        # Prevent duplicate entries
        if new_token_string in inverse_vocab:
            break

        vocab[current_vocab_size] = new_token_string
        inverse_vocab[new_token_string] = current_vocab_size

        # 5. Update ALL sequences by merging the occurrences of the best_pair
        for ticker in current_sequences.keys():
            sequence = current_sequences[ticker]
            new_sequence = []
            i = 0
            while i < len(sequence):
                if (
                    i < len(sequence) - 1
                    and sequence[i] == best_pair[0]
                    and sequence[i + 1] == best_pair[1]
                ):
                    new_sequence.append(new_token_string)
                    i += 2  # Skip the next element as it's merged
                else:
                    new_sequence.append(sequence[i])
                    i += 1
            current_sequences[ticker] = new_sequence

        current_vocab_size += 1

    # Map the final compressed token sequences to their respective integer IDs
    compressed_ticker_sequences = {}
    for ticker, sequence in current_sequences.items():
        compressed_ticker_sequences[ticker] = [
            inverse_vocab.get(tok, -1) for tok in sequence
        ]

    return compressed_ticker_sequences, vocab


# =====================================================================
# Execution Pipeline Block
# =====================================================================
if __name__ == "__main__":
    # Expand target vocab to accommodate the base tokens + the multi-day merges
    TARGET_VOCAB = 100 # Reduced to decrease the size of the transition matrix

    print("Running Global Native Time-Series BPE Compression Across All Tickers...")

    # Call the updated BPE function with the ticker_to_sequence from Step 1
    all_ticker_compressed_sequences, global_bpe_vocab = native_time_series_bpe(
        ticker_to_sequence, target_vocab_size=TARGET_VOCAB
    )

    print("\n==========================================")
    print("         GLOBAL BPE COMPRESSION METRICS     ")
    print("==========================================")

    total_original_tokens = sum(len(seq) for seq in ticker_to_sequence.values())
    total_compressed_tokens = sum(len(seq) for seq in all_ticker_compressed_sequences.values())

    print(f"Total Original Market Days Across All Tickers:  {total_original_tokens}")
    print(f"Total Compressed Sequence Length Across All Tickers: {total_compressed_tokens}")

    if total_compressed_tokens > 0:
        compression_ratio = total_original_tokens / total_compressed_tokens
        print(f"Compression Ratio:           {compression_ratio:.2f}x")

    print("\nLearned Multi-Day Transitions/Regimes:")
    has_merged_patterns = False
    for token_id, token_structure in global_bpe_vocab.items():
        if "➔" in token_structure:
            print(f"Token ID {token_id:03d}:  {token_structure}")
            has_merged_patterns = True

    if not has_merged_patterns:
        print(
            "No structural merges occurred. Check if your data has enough repeating candle tokens."
        )


Running Global Native Time-Series BPE Compression Across All Tickers...
Initial unique daily candle types found: 45

         GLOBAL BPE COMPRESSION METRICS     
Total Original Market Days Across All Tickers:  152189
Total Compressed Sequence Length Across All Tickers: 135913
Compression Ratio:           1.12x

Learned Multi-Day Transitions/Regimes:
Token ID 045:  DOML ➔ DOML
Token ID 046:  DOML ➔ DOLM
Token ID 047:  LGSS ➔ LLSS
Token ID 048:  MGMM ➔ DOML
Token ID 049:  MGMM ➔ DOLM
Token ID 050:  LGSS ➔ LGSS
Token ID 051:  DOLM ➔ DOML
Token ID 052:  MGMM ➔ MLMM
Token ID 053:  LGSS ➔ LLSM
Token ID 054:  LLSS ➔ LGSM
Token ID 055:  DOML ➔ MLMM
Token ID 056:  LGSS ➔ LGMS
Token ID 057:  LLMS ➔ LGSM
Token ID 058:  LGSS ➔ LGSM
Token ID 059:  DOLM ➔ DOLM
Token ID 060:  LLSM ➔ LGSM
Token ID 061:  LGSS ➔ MLSL
Token ID 062:  LGSS ➔ MLMM
Token ID 063:  DOSS ➔ DOSS
Token ID 064:  LGMS ➔ LLSS
Token ID 065:  DOLM ➔ MLMM
Token ID 066:  MGMM ➔ MGMM
Token ID 067:  LLSS ➔ LLSS
Token ID 068:  LGSS ➔ DOLM


In [32]:
# Step 3: Consolidated Transition Matrix and Scoring
def calculate_transition_matrix(
    all_ticker_compressed_sequences: dict[str, list[int]], vocab: dict
) -> pd.DataFrame:
    """Calculates the transition probability matrix from compressed sequences of token IDs
    across all tickers.
    """
    num_tokens = len(vocab)

    # 1. Initialize a raw count matrix (Rows = State T, Columns = State T+1)
    count_matrix = np.zeros((num_tokens, num_tokens))

    # Aggregate counts from all ticker sequences
    for ticker, compressed_ids in all_ticker_compressed_sequences.items():
        for t in range(len(compressed_ids) - 1):
            current_state = compressed_ids[t]
            next_state = compressed_ids[t + 1]
            # Only count if both states are valid token IDs
            if 0 <= current_state < num_tokens and 0 <= next_state < num_tokens:
                count_matrix[current_state, next_state] += 1

    # 2. Convert counts to row-normalized conditional probabilities: P(T+1 | T)
    # Handle rows with zero counts (unvisited tokens) to avoid division by zero
    row_sums = count_matrix.sum(axis=1, keepdims=True)
    prob_matrix = np.where(row_sums > 0, count_matrix / row_sums, 0)

    # 3. Format into a clean, readable DataFrame with token descriptions as labels
    token_labels = [f"[{i:03d}] {vocab[i]}" for i in range(num_tokens)]

    df_prob = pd.DataFrame(
        prob_matrix, index=token_labels, columns=token_labels
    )

    return df_prob


def extract_trading_edges(
    prob_df: pd.DataFrame, min_probability: float = 0.35
):
    """Filters the transition matrix to uncover highly recurring probabilities

    that could form the basis of a 1-to-5 day trade setup.
    """
    edges = []

    for current_state in prob_df.index:
        for next_state in prob_df.columns:
            prob = prob_df.loc[current_state, next_state]

            # We care about transitions that are highly probable and not just noise
            if prob >= min_probability and current_state != next_state:
                # Basic check to ensure the state actually exists in historical data
                edges.append(
                    {
                        "Current Regime (T)": current_state,
                        "Next Regime (T+1)": next_state,
                        "Transition Probability": f"{prob * 100:.1f}%",
                    }
                )

    return pd.DataFrame(edges)


# =====================================================================
# Execution Block
# =====================================================================
if __name__ == "__main__":
    print("Calculating Step 3 Global Transition Matrix...")

    # 1. Compute the structural global transition matrix using all ticker sequences
    global_transitions = calculate_transition_matrix(
        all_ticker_compressed_sequences, global_bpe_vocab
    )

    # 2. Extract highly probable transitions
    # (Adjust min_probability based on how strict you want your signal filter to be)
    predictive_edges = extract_trading_edges(
        global_transitions, min_probability=0.05 # Lowered min_probability
    )

    print("\n=========================================")
    print("     EXTRACTED GLOBAL MULTI-DAY REGIME EDGES     ")
    print("=========================================")
    if not predictive_edges.empty:
        # Sort by highest probability edge
        predictive_edges = predictive_edges.sort_values(
            by="Transition Probability", ascending=False
        )
        print(predictive_edges.to_string(index=False))
    else:
        print(
            "No transitions met the minimum probability threshold. Try lowering min_probability."
        )


Calculating Step 3 Global Transition Matrix...

     EXTRACTED GLOBAL MULTI-DAY REGIME EDGES     
Current Regime (T) Next Regime (T+1) Transition Probability
        [005] DOMS        [001] DOLM                   9.7%
        [004] DOMM        [006] DOSL                   9.5%
        [007] DOSM        [008] DOSS                   9.2%
 [063] DOSS ➔ DOSS        [002] DOLS                   8.9%
 [076] MGMS ➔ DOML        [041] MLMS                   8.7%
        [005] DOMS        [006] DOSL                   8.7%
        [007] DOSM        [043] MLSM                   8.3%
        [017] LGSS        [038] MLLS                   8.0%
        [017] LGSS        [022] LLMM                   8.0%
 [054] LLSS ➔ LGSM        [026] LLSS                   8.0%
        [027] MGLL        [016] LGSM                   8.0%
        [005] DOMS        [008] DOSS                   7.8%
        [005] DOMS        [002] DOLS                   7.8%
 [071] DOML ➔ MGMM        [042] MLSL                   7.7%
 [

In [33]:
# Optimize and score trading edges
def optimize_and_score_edges(
    edges_df: pd.DataFrame, min_sample_size: int = 5
) -> pd.DataFrame:
    """Takes the output of extract_trading_edges, classifies the directional bias

    of the target states, and ranks them by actionable trading conviction.
    """
    if edges_df.empty:
        print("No edges to optimize.")
        return edges_df

    # Group by Current Regime (T) to determine an overall bias for each setup
    aggregated_signals = []
    for current_setup, group in edges_df.groupby("Current Regime (T)"):
        bullish_prob_sum = 0.0
        bearish_prob_sum = 0.0

        for _, row in group.iterrows():
            next_state = row["Next Regime (T+1)"]
            prob_str = row["Transition Probability"]

            # Parse the text descriptors to extract the raw candle strings
            target_token_desc = next_state.split("]")[-1].strip()
            if "➔" in target_token_desc:
                target_candle = target_token_desc.split("➔")[-1].strip()
            else:
                target_candle = target_token_desc

            # Extract the 2-letter body code prefix (LL, ML, DO, MG, LG)
            body_code = target_candle[:2]

            # Convert probability string back to a float
            prob_float = float(prob_str.replace("%", "")) / 100.0

            if body_code in ["LG", "MG"]:
                bullish_prob_sum += prob_float
            elif body_code in ["LL", "ML"]:
                bearish_prob_sum += prob_float

        # Determine overall directional bias for the current setup
        directional_bias = "NEUTRAL / REVERSION"
        conviction_score = 0.0

        if bullish_prob_sum > bearish_prob_sum:
            directional_bias = "BULLISH LONG"
            conviction_score = bullish_prob_sum - bearish_prob_sum
        elif bearish_prob_sum > bullish_prob_sum:
            directional_bias = "BEARISH SHORT"
            conviction_score = bearish_prob_sum - bullish_prob_sum # Conviction towards a bearish move

        if directional_bias != "NEUTRAL / REVERSION":
            aggregated_signals.append(
                {
                    "Current Setup (T)": current_setup,
                    "Directional Bias": directional_bias,
                    "Bullish Probability": f"{bullish_prob_sum * 100:.1f}%",
                    "Bearish Probability": f"{bearish_prob_sum * 100:.1f}%",
                    "Conviction Score": round(conviction_score, 3),
                }
            )

    # Convert to DataFrame
    opt_df = pd.DataFrame(aggregated_signals)

    # Sort by the absolute strength of the Conviction Score (if you want to prioritize stronger signals)
    opt_df = opt_df.sort_values(by="Conviction Score", ascending=False)

    return opt_df


# =====================================================================
# Execution Block (Run right below your current script)
# =====================================================================
if __name__ == "__main__":
    print("\n=========================================")
    print("     OPTIMIZED DIRECTIONAL SWING TRADES    ")
    print("=========================================")

    # Pass the 'predictive_edges' DataFrame generated by your code directly into the optimizer
    optimized_signals = optimize_and_score_edges(predictive_edges)

    if not optimized_signals.empty:
        print(
            optimized_signals[
                [
                    "Current Setup (T)",
                    "Directional Bias",
                    "Bullish Probability",
                    "Bearish Probability",
                    "Conviction Score",
                ]
            ].to_string(index=False)
        )
    else:
        print(
            "No directional edges found. All high-probability transitions lead to neutral/Doji states."
        )



     OPTIMIZED DIRECTIONAL SWING TRADES    
Current Setup (T) Directional Bias Bullish Probability Bearish Probability  Conviction Score
       [009] LGLL     BULLISH LONG               62.5%               37.5%             0.250
[088] MGMM ➔ MLSM    BEARISH SHORT                0.0%               18.2%             0.182
[084] LGSS ➔ LLMS    BEARISH SHORT                5.2%               23.2%             0.180
[091] LGSS ➔ MGLM    BEARISH SHORT                0.0%               18.0%             0.180
[056] LGSS ➔ LGMS    BEARISH SHORT                0.0%               17.0%             0.170
[062] LGSS ➔ MLMM    BEARISH SHORT                5.2%               22.0%             0.168
[058] LGSS ➔ LGSM    BEARISH SHORT                0.0%               16.2%             0.162
       [018] LLLL    BEARISH SHORT               42.9%               57.2%             0.143
[054] LLSS ➔ LGSM    BEARISH SHORT                6.0%               18.8%             0.128
       [010] LGLM    BEAR

In [34]:
# Step 4: Multi-ticker forward-return validator
def validate_forward_returns(
    stacked_data: pd.DataFrame,
    all_ticker_compressed_sequences: dict[str, list[int]],
    vocab: dict,
    target_signals_df: pd.DataFrame,
    original_ticker_sequences: dict[str, list[str]] # Added to correctly map BPE tokens to original dates
) -> pd.DataFrame:
    """Matches optimized BPE signals back to historical trading dates and calculates

    the actual 1-day, 3-day, and 5-day forward percentage returns for each ticker.
    """
    if target_signals_df.empty:
        print("No signals available to validate.")
        return pd.DataFrame()

    # Store raw data for aggregation later
    raw_performance_data = []

    # Create a quick reverse lookup dictionary for perfect string matching
    # Strips any stray whitespaces to guarantee a clean match
    string_to_id = {
        token_str.strip(): token_id
        for token_id, token_str in vocab.items()
    }

    # Iterate through each ticker to perform validation
    for ticker, compressed_ids_for_ticker in all_ticker_compressed_sequences.items():
        # Filter stacked_data for the current ticker and create a copy to avoid SettingWithCopyWarning
        df_ticker = stacked_data.loc[pd.IndexSlice[:, ticker], :].copy()
        df_ticker.index = df_ticker.index.droplevel('Ticker') # Drop Ticker level for easier indexing

        if df_ticker.empty or len(compressed_ids_for_ticker) == 0:
            continue

        # Ensure df_ticker is sorted by Date before calculating shifts
        df_ticker = df_ticker.sort_index()

        # Calculate forward returns for the current ticker
        df_ticker["Forward_1D"] = df_ticker["Close"].shift(-1) / df_ticker["Close"] - 1
        df_ticker["Forward_3D"] = df_ticker["Close"].shift(-3) / df_ticker["Close"] - 1
        df_ticker["Forward_5D"] = df_ticker["Close"].shift(-5) / df_ticker["Close"] - 1

        # Reconstruct the date mapping for the BPE tokens for this specific ticker
        # Each BPE token corresponds to the *end date* of the pattern it represents in the original sequence.
        timeline_dates = []
        original_idx_counter = 0 # Tracks position in the original, uncompressed sequence

        # Get the original candle token sequence for this ticker to correctly calculate 'days_spanned'
        original_sequence_for_ticker = original_ticker_sequences.get(ticker, [])

        for compressed_id_val in compressed_ids_for_ticker:
            token_desc = vocab.get(compressed_id_val, '')
            # Calculate days spanned by the *original* pattern that led to this compressed token
            # This logic assumes a direct positional mapping where `days_spanned` reflects the number of single tokens merged.
            days_spanned = len(token_desc.split(" ➔ ")) if " ➔ " in token_desc else 1

            # Ensure trigger_index does not go out of bounds of the original ticker's DataFrame
            trigger_index = original_idx_counter + days_spanned - 1
            if trigger_index < len(df_ticker):
                timeline_dates.append(df_ticker.index[trigger_index])
            else:
                # Fallback: if the BPE sequence is longer than expected, or misaligned
                # This case indicates a potential issue in sequence generation/alignment
                timeline_dates.append(df_ticker.index[-1]) # Use last available date as a best guess

            original_idx_counter += days_spanned

        # Create timeline_df for the current ticker
        # This DataFrame maps the *end date* of a BPE pattern to its Token_ID
        # It will have the same length as `compressed_ids_for_ticker`
        timeline_df_for_ticker = pd.DataFrame(
            {"Date": timeline_dates[:len(compressed_ids_for_ticker)], "Token_ID": compressed_ids_for_ticker}
        ).set_index("Date")

        # Now, proceed with matching and calculating performance for this ticker
        for _, signal in target_signals_df.iterrows():
            setup_desc = signal["Setup Pattern (T)"].strip()
            bias = signal["Directional Bias"]

            setup_id = None
            if setup_desc in string_to_id:
                setup_id = string_to_id[setup_desc]
            else:
                try:
                    setup_id = int(
                        signal["Setup Pattern (T)"].split("]")[0].replace("[", "")
                    )
                except ValueError:
                    continue # Skip if setup_id cannot be parsed

            if setup_id is None:
                continue

            # Locate all calendar dates for this specific setup in the current ticker's timeline
            trigger_dates = timeline_df_for_ticker[timeline_df_for_ticker["Token_ID"] == setup_id].index

            # Extract the forward returns specifically for those dates from the current ticker's data
            matched_returns = df_ticker.loc[df_ticker.index.isin(trigger_dates)]

            if len(matched_returns) == 0:
                continue

            # Adjust returns based on directional bias for aggregation
            if bias == "BULLISH LONG":
                # For bullish long, positive forward returns are gains, negative are losses
                actual_1d_returns = matched_returns["Forward_1D"]
                actual_3d_returns = matched_returns["Forward_3D"]
                actual_5d_returns = matched_returns["Forward_5D"]
            elif bias == "BEARISH SHORT":
                # For bearish short, negative forward returns are gains, positive are losses (inverted)
                actual_1d_returns = -matched_returns["Forward_1D"]
                actual_3d_returns = -matched_returns["Forward_3D"]
                actual_5d_returns = -matched_returns["Forward_5D"]
            else:
                # Should not happen with filtered directional_plays, but as a safeguard
                actual_1d_returns = pd.Series([0.0] * len(matched_returns))
                actual_3d_returns = pd.Series([0.0] * len(matched_returns))
                actual_5d_returns = pd.Series([0.0] * len(matched_returns))

            # Store raw data for aggregation later
            num_wins_5d = ((matched_returns["Forward_5D"] > 0).sum() if bias == "BULLISH LONG" else (matched_returns["Forward_5D"] < 0).sum()) if not matched_returns.empty else 0
            num_losses_5d = ((matched_returns["Forward_5D"] <= 0).sum() if bias == "BULLISH LONG" else (matched_returns["Forward_5D"] >= 0).sum()) if not matched_returns.empty else 0 # Careful: >=0 for short losses


            raw_performance_data.append(
                {
                    "Setup Pattern (T)": setup_desc,
                    "Direction": bias,
                    "Sum 1D Returns": actual_1d_returns.sum(),
                    "Sum 3D Returns": actual_3d_returns.sum(),
                    "Sum 5D Returns": actual_5d_returns.sum(),
                    "Sum 5D Winning Returns": actual_5d_returns[actual_5d_returns > 0].sum(), # New
                    "Sum 5D Losing Returns": actual_5d_returns[actual_5d_returns <= 0].sum(), # New, will be negative
                    "Num Wins 5D": num_wins_5d,
                    "Num Losses 5D": num_losses_5d, # New
                    "Sample Count": len(matched_returns),
                }
            )

    # Aggregate the raw performance data
    if not raw_performance_data:
        return pd.DataFrame()

    temp_df = pd.DataFrame(raw_performance_data)
    aggregated_df = temp_df.groupby(['Setup Pattern (T)', 'Direction']).agg(
        Total_Sample_Count=('Sample Count', 'sum'),
        Sum_1D_Returns=('Sum 1D Returns', 'sum'),
        Sum_3D_Returns=('Sum 3D Returns', 'sum'),
        Sum_5D_Returns=('Sum 5D Returns', 'sum'),
        Sum_5D_Winning_Returns=('Sum 5D Winning Returns', 'sum'), # New
        Sum_5D_Losing_Returns=('Sum 5D Losing Returns', 'sum'), # New
        Total_Wins_5D=('Num Wins 5D', 'sum'),
        Total_Losses_5D=('Num Losses 5D', 'sum') # New
    ).reset_index()

    # Calculate average returns and win rates
    aggregated_df['Avg 1D Return'] = aggregated_df['Sum_1D_Returns'] / aggregated_df['Total_Sample_Count']
    aggregated_df['Avg 3D Return'] = aggregated_df['Sum_3D_Returns'] / aggregated_df['Total_Sample_Count']
    aggregated_df['Avg 5D Return'] = aggregated_df['Sum_5D_Returns'] / aggregated_df['Total_Sample_Count']
    aggregated_df['5D Win Rate'] = aggregated_df['Total_Wins_5D'] / aggregated_df['Total_Sample_Count'] if aggregated_df['Total_Sample_Count'].sum() > 0 else 0

    # Calculate Gross Profit (sum of all positive returns)
    aggregated_df['Gross Profit 5D'] = aggregated_df['Sum_5D_Winning_Returns']
    # Calculate Gross Loss (absolute sum of all negative returns)
    aggregated_df['Gross Loss 5D'] = aggregated_df['Sum_5D_Losing_Returns'].abs()

    # Profit Factor
    aggregated_df['Profit Factor 5D'] = aggregated_df.apply(
        lambda row: row['Gross Profit 5D'] / row['Gross Loss 5D'] if row['Gross Loss 5D'] > 0 else np.inf, axis=1
    )

    # Expectancy (Average Profit per Trade)
    aggregated_df['Expectancy 5D'] = aggregated_df['Sum_5D_Returns'] / aggregated_df['Total_Sample_Count']

    # Average Win / Average Loss Ratio
    aggregated_df['Avg Win 5D'] = aggregated_df.apply(
        lambda row: row['Gross Profit 5D'] / row['Total_Wins_5D'] if row['Total_Wins_5D'] > 0 else 0, axis=1
    )
    aggregated_df['Avg Loss 5D'] = aggregated_df.apply(
        lambda row: row['Gross Loss 5D'] / row['Total_Losses_5D'] if row['Total_Losses_5D'] > 0 else 0, axis=1
    )
    aggregated_df['Reward/Risk Ratio 5D'] = aggregated_df.apply(
        lambda row: row['Avg Win 5D'] / row['Avg Loss 5D'] if row['Avg Loss 5D'] > 0 else np.inf, axis=1
    )

    # Rename columns for final output
    aggregated_df = aggregated_df.rename(columns={'Total_Sample_Count': 'Sample Count'})

    # Select desired columns for the final report
    final_cols = [
        'Setup Pattern (T)', 'Direction', 'Sample Count',
        'Avg 1D Return', 'Avg 3D Return', 'Avg 5D Return', '5D Win Rate',
        'Profit Factor 5D', 'Expectancy 5D', 'Avg Win 5D', 'Avg Loss 5D', 'Reward/Risk Ratio 5D'
    ]
    return aggregated_df[final_cols]


# =====================================================================
# Performance Validation Execution Block
# =====================================================================
if __name__ == "__main__":
    print("Executing Step 4: Validating Historical Forward Returns Across All Tickers...")

    # Rename 'Current Setup (T)' to 'Setup Pattern (T)' in optimized_signals for consistency
    optimized_signals_renamed = optimized_signals.rename(columns={'Current Setup (T)': 'Setup Pattern (T)'})

    # Call the updated validation function with multi-ticker data
    performance_report = validate_forward_returns(
        stacked_data, all_ticker_compressed_sequences, global_bpe_vocab, optimized_signals_renamed, ticker_to_sequence
    )

    print("\n=======================================================================")
    print("                   FORWARD-RETURN PERFORMANCE VALIDATION               ")
    print("=======================================================================")
    if not performance_report.empty:
        print(f"Total number of lines in performance_report: {len(performance_report)}")

        filtered_performance_report = performance_report[performance_report['Sample Count'] > 30].copy()
        print(f"Total number of lines in filtered performance_report: {len(filtered_performance_report)}")

        # These are now floats from the function, no need for .str.rstrip('%')
        filtered_performance_report = filtered_performance_report.sort_values(by='Avg 5D Return', ascending=False)

        if not filtered_performance_report.empty:
            # Format the output for display
            display_df = filtered_performance_report.copy()
            display_df['Avg 1D Return'] = display_df['Avg 1D Return'].apply(lambda x: f"{x * 100:+.2f}%")
            display_df['Avg 3D Return'] = display_df['Avg 3D Return'].apply(lambda x: f"{x * 100:+.2f}%")
            display_df['Avg 5D Return'] = display_df['Avg 5D Return'].apply(lambda x: f"{x * 100:+.2f}%")
            display_df['5D Win Rate'] = display_df['5D Win Rate'].apply(lambda x: f"{x * 100:.1f}%")
            display_df['Profit Factor 5D'] = display_df['Profit Factor 5D'].apply(lambda x: f"{x:.2f}" if np.isfinite(x) else "N/A")
            display_df['Expectancy 5D'] = display_df['Expectancy 5D'].apply(lambda x: f"{x * 100:+.2f}%")
            display_df['Avg Win 5D'] = display_df['Avg Win 5D'].apply(lambda x: f"{x * 100:+.2f}%")
            display_df['Avg Loss 5D'] = display_df['Avg Loss 5D'].apply(lambda x: f"{x * 100:+.2f}%")
            display_df['Reward/Risk Ratio 5D'] = display_df['Reward/Risk Ratio 5D'].apply(lambda x: f"{x:.2f}" if np.isfinite(x) else "N/A")

            print(display_df.to_string(index=False))
        else:
            print("No signals found with Sample Count > 30.")
    else:
        print(
            "No historical trade instances could be verified. Check string matching format or data availability."
        )


Executing Step 4: Validating Historical Forward Returns Across All Tickers...

                   FORWARD-RETURN PERFORMANCE VALIDATION               
Total number of lines in performance_report: 79
Total number of lines in filtered performance_report: 77
Setup Pattern (T)     Direction  Sample Count Avg 1D Return Avg 3D Return Avg 5D Return 5D Win Rate Profit Factor 5D Expectancy 5D Avg Win 5D Avg Loss 5D Reward/Risk Ratio 5D
       [019] LLLM  BULLISH LONG           562        +1.08%        +2.01%        +3.48%       54.6%             1.89        +3.48%    +13.49%      +8.61%                 1.57
[098] LLSM ➔ LGMS  BULLISH LONG           224        +0.94%        +1.76%        +2.10%       59.4%             2.06        +2.10%     +6.86%      +4.86%                 1.41
       [036] MLLL  BULLISH LONG           365        +0.22%        +1.19%        +2.05%       55.6%             1.82        +2.05%     +8.16%      +5.67%                 1.44
[060] LLSM ➔ LGSM  BULLISH LONG           33

In [35]:
lglm_rows = performance_report[performance_report['Setup Pattern (T)'] == '[022] LLMM']
display(lglm_rows)

,Setup Pattern (T),Direction,Sample Count,Avg 1D Return,Avg 3D Return,Avg 5D Return,5D Win Rate,Profit Factor 5D,Expectancy 5D,Avg Win 5D,Avg Loss 5D,Reward/Risk Ratio 5D
17,[022] LLMM,BULLISH LONG,3665,0.000666,0.004805,0.010951,0.53779,1.358622,0.010951,0.077146,0.06646,1.160791


### Filter for High-Conviction Signals

Let's identify signals from our `performance_report` that have an absolute `Avg 5D Return` of 1.5% or greater. These will be considered our `high_conviction_signals` for out-of-sample testing.

In [36]:
# Filter for signals with an absolute Avg 5D Return >= 1.5%
high_conviction_signals = filtered_performance_report[
    filtered_performance_report['Avg 5D Return'].abs() >= 1.5
].copy()

print("High-Conviction Signals (Abs Avg 5D Return >= 1.5%):")
if not high_conviction_signals.empty:
    display(high_conviction_signals)
else:
    print("No high-conviction signals found with Abs Avg 5D Return >= 1.5%.")

High-Conviction Signals (Abs Avg 5D Return >= 1.5%):
No high-conviction signals found with Abs Avg 5D Return >= 1.5%.


### Out-of-Sample Data Fetch and Candlestick Tokenization

Now, we'll download a new, out-of-sample dataset using the `END_DATE` of our training data as the start date, and two weeks prior to the current date as the new end date. This ensures we are testing on unseen data.

In [37]:
from datetime import datetime, timedelta

# Define out-of-sample date range
START_DATE_OOS = END_DATE # Use the end date of the in-sample data as the start for OOS
END_DATE_OOS = (datetime.now() - timedelta(weeks=2)).strftime('%Y-%m-%d')

print(f"Downloading out-of-sample data for {len(symbols_list)} symbols from {START_DATE_OOS} to {END_DATE_OOS}...")

# Download out-of-sample raw data
raw_data_oos = yf.download(
    tickers=symbols_list,
    start=START_DATE_OOS,
    end=END_DATE_OOS,
    interval="1d",
    auto_adjust=True,
)

# Stack columns for multi-index format
if isinstance(raw_data_oos.columns, pd.MultiIndex):
    if "Ticker" in raw_data_oos.columns.names:
        stacked_data_oos = raw_data_oos.stack(level="Ticker")
    else:
        stacked_data_oos = raw_data_oos.stack(level=1)
else:
    # Handle case if only a single ticker is downloaded, or an error occurred
    print("Warning: raw_data_oos is not multi-indexed, potentially single ticker or download error.")
    stacked_data_oos = raw_data_oos.copy()
    if 'Ticker' not in stacked_data_oos.columns and 'Adj Close' in stacked_data_oos.columns:
        stacked_data_oos['Ticker'] = symbols_list[0] # Assuming first symbol if single
        stacked_data_oos = stacked_data_oos.set_index(['Ticker'], append=True)
    elif 'Adj Close' not in stacked_data_oos.columns and not stacked_data_oos.empty:
        # Likely an error message from yfinance, or no data. Handle appropriately.
        print("No price data found for out-of-sample period or unexpected data format.")
        stacked_data_oos = pd.DataFrame() # Set to empty if truly no data

# Drop missing values after stacking
if not stacked_data_oos.empty:
    stacked_data_oos = stacked_data_oos.dropna(subset=["Open", "High", "Low", "Close"])
    print(f"Out-of-sample data downloaded and processed. Shape: {stacked_data_oos.shape}")
    print("Generating Candlestick Tokens for out-of-sample data...")
    stacked_data_oos["Candle_Token"] = generate_candlestick_tokens(stacked_data_oos)

    # Reconstruct clean individual timeline lists for OOS BPE application
    ticker_to_sequence_oos = {}
    for ticker, group in stacked_data_oos["Candle_Token"].groupby(level="Ticker"):
        chronological_seq_oos = group.droplevel("Ticker").sort_index()
        ticker_to_sequence_oos[ticker] = chronological_seq_oos.astype(str).tolist()

    print(f"--- Out-of-Sample Framework Initialization Complete: {len(ticker_to_sequence_oos)} Active Timelines Set ---")
    display(stacked_data_oos[["Open", "High", "Low", "Close", "Candle_Token"]].head())
else:
    print("Skipping OOS tokenization due to empty out-of-sample data.")
    ticker_to_sequence_oos = {}

[*********************100%***********************]  100 of 100 completed
/tmp/ipykernel_3057/2208266491.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked_data_oos = raw_data_oos.stack(level="Ticker")


Out-of-sample data downloaded and processed. Shape: (35206, 5)
Generating Candlestick Tokens for out-of-sample data...
--- Out-of-Sample Framework Initialization Complete: 100 Active Timelines Set ---


Price                    Open        High         Low       Close Candle_Token
Date       Ticker                                                             
2024-12-31 AAOI     36.400002   37.700001   35.959999   36.860001         MGLM
           AAPL    250.837411  251.672075  247.846510  248.830231         MLMM
           ADBE    446.350006  448.500000  442.809998  444.679993         MLMM
           AMAT    162.395653  163.136374  159.966097  160.617935         MLMM
           AMD     123.099998  123.550003  120.139999  120.790001         LLSM

### Apply Previously Learned BPE Vocabulary to Out-of-Sample Data

We need a function to apply the BPE vocabulary learned from the in-sample data to the new out-of-sample sequences. This ensures we are not learning new patterns from the future data, which would introduce look-ahead bias.

In [38]:
def apply_global_bpe_to_new_sequences(
    new_ticker_to_sequence: dict[str, list[str]], global_bpe_vocab: dict
) -> dict[str, list[int]]:
    """Applies a pre-trained global BPE vocabulary to new ticker sequences.

    Args:
        new_ticker_to_sequence: A dictionary of new ticker sequences (raw candle tokens).
        global_bpe_vocab: The pre-trained BPE vocabulary (token_id -> token_string).

    Returns:
        A dictionary of compressed ticker sequences (ticker -> list of token_ids).
    """
    # Create inverse mapping (token string -> token ID) for efficient lookup
    inverse_global_bpe_vocab = {
        token_str: token_id for token_id, token_str in global_bpe_vocab.items()
    }

    # Extract all merged patterns from the vocabulary.
    merged_patterns_list = []
    for token_id, token_str in global_bpe_vocab.items():
        if " ➔ " in token_str:
            parts = token_str.split(" ➔ ")
            merged_patterns_list.append((token_str, parts, token_id))

    # Sort merged patterns by the length of their constituent parts in descending order.
    merged_patterns_list.sort(key=lambda x: len(x[1]), reverse=True)

    all_ticker_compressed_sequences_oos = {}

    for ticker, sequence_of_strings in new_ticker_to_sequence.items():
        current_sequence = list(sequence_of_strings)  # Start with raw tokens

        # Apply the merges iteratively for a few passes.
        for _ in range(len(merged_patterns_list) + 1): # Max passes = num merges + 1
            changed_this_pass = False
            new_sequence_for_pass = []
            i = 0
            while i < len(current_sequence):
                best_match_len = 0
                best_match_token_str = None

                # Find the longest matching merged pattern starting at current_sequence[i]
                for merged_token_str, parts, _ in merged_patterns_list:
                    if len(parts) <= (len(current_sequence) - i) and \
                       all(current_sequence[i + j] == parts[j] for j in range(len(parts))):
                        if len(parts) > best_match_len:
                            best_match_len = len(parts)
                            best_match_token_str = merged_token_str

                if best_match_len > 0 and best_match_token_str is not None:
                    new_sequence_for_pass.append(best_match_token_str)
                    i += best_match_len
                    changed_this_pass = True
                else:
                    new_sequence_for_pass.append(current_sequence[i])
                    i += 1

            if not changed_this_pass and len(new_sequence_for_pass) == len(current_sequence):
                break # No changes made in this pass, stop early

            current_sequence = new_sequence_for_pass

        # After applying all possible merges, convert the final sequence of token strings
        compressed_ids = []
        for token_str in current_sequence:
            token_id = inverse_global_bpe_vocab.get(token_str)
            if token_id is None:
                print(f"Warning: Unknown token '{token_str}' encountered for ticker {ticker} in OOS. Assigning -1.")
                compressed_ids.append(-1) # Placeholder for unknown tokens
            else:
                compressed_ids.append(token_id)

        all_ticker_compressed_sequences_oos[ticker] = compressed_ids

    return all_ticker_compressed_sequences_oos


print("Applying learned BPE vocabulary to out-of-sample sequences...")
if ticker_to_sequence_oos:
    all_ticker_compressed_sequences_oos = apply_global_bpe_to_new_sequences(
        ticker_to_sequence_oos, global_bpe_vocab
    )
    print(f"Successfully compressed {len(all_ticker_compressed_sequences_oos)} out-of-sample ticker sequences.")
else:
    print("No out-of-sample data to compress.")
    all_ticker_compressed_sequences_oos = {}

Applying learned BPE vocabulary to out-of-sample sequences...
Successfully compressed 100 out-of-sample ticker sequences.


### Validate High-Conviction Signals on Out-of-Sample Data

Finally, we'll use the `validate_forward_returns` function to evaluate the performance of our `high_conviction_signals` on the out-of-sample data.

In [39]:
if not high_conviction_signals.empty and not stacked_data_oos.empty and all_ticker_compressed_sequences_oos:
    print("Validating High-Conviction Signals on Out-of-Sample Data...")

    # Rename 'Current Setup (T)' to 'Setup Pattern (T)' and 'Direction' to 'Directional Bias' for consistency
    high_conviction_signals_renamed = high_conviction_signals.rename(columns={'Current Setup (T)': 'Setup Pattern (T)', 'Direction': 'Directional Bias'})

    oos_performance_report = validate_forward_returns(
        stacked_data_oos,
        all_ticker_compressed_sequences_oos,
        global_bpe_vocab,
        high_conviction_signals_renamed, # Pass the renamed high-conviction signals
        ticker_to_sequence_oos
    )

    print("\n=======================================================================")
    print("           OUT-OF-SAMPLE FORWARD-RETURN PERFORMANCE VALIDATION       ")
    print("=======================================================================")
    if not oos_performance_report.empty:
        # Merge with high_conviction_signals_renamed to add 'Expected 5D Return'
        # First, ensure column names match for merging
        oos_performance_report_for_merge = oos_performance_report.rename(columns={'Direction': 'Directional Bias'})

        oos_performance_report = pd.merge(
            oos_performance_report_for_merge,
            high_conviction_signals_renamed[['Setup Pattern (T)', 'Directional Bias', 'Avg 5D Return']],
            on=['Setup Pattern (T)', 'Directional Bias'],
            how='left',
            suffixes=('', '_expected')
        )
        oos_performance_report.rename(columns={'Avg 5D Return_expected': 'Expected 5D Return'}, inplace=True)

        # 'Expected 5D Return' from the merge will be float, no need to convert.
        # Other return/win rate columns are now floats from validate_forward_returns, so no .str.rstrip('%') needed.

        oos_performance_report = oos_performance_report.sort_values(by='Avg 5D Return', ascending=False)

        # Format the output for display
        display_oos_df = oos_performance_report.copy()
        display_oos_df['Avg 1D Return'] = display_oos_df['Avg 1D Return'].apply(lambda x: f"{x * 100:+.2f}%")
        display_oos_df['Avg 3D Return'] = display_oos_df['Avg 3D Return'].apply(lambda x: f"{x * 100:+.2f}%")
        display_oos_df['Avg 5D Return'] = display_oos_df['Avg 5D Return'].apply(lambda x: f"{x * 100:+.2f}%")
        display_oos_df['5D Win Rate'] = display_oos_df['5D Win Rate'].apply(lambda x: f"{x * 100:.1f}%")
        display_oos_df['Expected 5D Return'] = display_oos_df['Expected 5D Return'].apply(lambda x: f"{x * 100:+.2f}%")
        display_oos_df['Profit Factor 5D'] = display_oos_df['Profit Factor 5D'].apply(lambda x: f"{x:.2f}" if np.isfinite(x) else "N/A")
        display_oos_df['Expectancy 5D'] = display_oos_df['Expectancy 5D'].apply(lambda x: f"{x * 100:+.2f}%")
        display_oos_df['Avg Win 5D'] = display_oos_df['Avg Win 5D'].apply(lambda x: f"{x * 100:+.2f}%")
        display_oos_df['Avg Loss 5D'] = display_oos_df['Avg Loss 5D'].apply(lambda x: f"{x * 100:+.2f}%")
        display_oos_df['Reward/Risk Ratio 5D'] = display_oos_df['Reward/Risk Ratio 5D'].apply(lambda x: f"{x:.2f}" if np.isfinite(x) else "N/A")


        print(display_oos_df.to_string(index=False))
    else:
        print("No out-of-sample trade instances could be verified for high-conviction signals.")
else:
    print("Skipping out-of-sample validation: either no high-conviction signals, no OOS data, or no compressed OOS sequences.")


Skipping out-of-sample validation: either no high-conviction signals, no OOS data, or no compressed OOS sequences.
